In [1]:
# ============================================
# 1. SETUP Y CARGA DEL MODELO DESDE GOOGLE DRIVE
# ============================================

import os
import re
import json
import math
import collections
import pandas as pd
import torch.nn.functional as F
import torch
import torch.nn as nn
from google.colab import drive

# 1. Montar Google Drive
drive.mount("/content/drive")
PROJ = "/content/drive/MyDrive/Proyecto_traductor"
DATA_DIR = os.path.join(PROJ, "data")
SPLIT_DIR = os.path.join(DATA_DIR, "splits")
VOCAB_DIR = os.path.join(PROJ, "vocabularios")
MODEL_DIR = os.path.join(PROJ, "modelos")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de inferencia: {DEVICE}")

# 2. Cargar Vocabularios y Configuraciones
with open(os.path.join(VOCAB_DIR, "vocab_english.json"), "r", encoding="utf-8") as f:
    stoi_en = json.load(f)

with open(os.path.join(VOCAB_DIR, "vocab_spanish.json"), "r", encoding="utf-8") as f:
    stoi_es = json.load(f)

with open(os.path.join(VOCAB_DIR, "itos_spanish.json"), "r", encoding="utf-8") as f:
    itos_es = json.load(f)

with open(os.path.join(MODEL_DIR, "model_config.json"), "r", encoding="utf-8") as f:
    model_cfg = json.load(f)

# Reconstruir listas de indices a palabras
itos_en = {v: k for k, v in stoi_en.items()}
itos_es = {v: k for k, v in stoi_es.items()}

PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

# 3. Tokenizador por reglas
TOK_RE = re.compile(r"\w+(?:'\w+)*|[^\w\s]")
def tokenize(text):
    return TOK_RE.findall(str(text))

# 4. Reconstrucción de la Arquitectura del Transformer (Desde Cero)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512,
                 n_layers=3, dropout=0.1, max_len=100, pad_idx=PAD_IDX):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)

# 5. Instanciar y Cargar Pesos
model = Transformer(
    src_vocab_size=len(stoi_en),
    tgt_vocab_size=len(stoi_es),
    d_model=model_cfg["d_model"],
    n_heads=model_cfg["n_heads"],
    d_ff=model_cfg["d_ff"],
    n_layers=model_cfg["n_layers"],
    dropout=model_cfg["dropout"],
    max_len=model_cfg["max_len"],
    pad_idx=PAD_IDX
).to(DEVICE)

CKPT_PATH = os.path.join(MODEL_DIR, "best_model.pth")
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

print(f"✅ ¡Modelo 'best_model.pth' cargado correctamente en {DEVICE}!")

Mounted at /content/drive
Dispositivo de inferencia: cuda
✅ ¡Modelo 'best_model.pth' cargado correctamente en cuda!


# Función de Traducción Autoregresiva (Greedy Decoding)

In [2]:
# ============================================
# 2. INFERENCIA CON BEAM SEARCH (K=3)
# ============================================

def translate_beam_search(sentence, beam_size=3, max_len=30):
    model.eval()

    # 1. Tokenizar y codificar frase de entrada
    tokens = tokenize(sentence)[:max_len]
    src_ids = [stoi_en.get(t, UNK_IDX) for t in tokens]
    src_ids = [SOS_IDX] + src_ids + [EOS_IDX]

    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)
    src_mask = model.make_src_mask(src_tensor)

    # 2. Salida del Encoder
    with torch.no_grad():
        enc_out = model.encoder(src_tensor, src_mask)

    # 3. Inicializar Beam: (secuencia_ids, log_prob_acumulada)
    beams = [([SOS_IDX], 0.0)]

    for _ in range(max_len):
        candidates = []
        all_completed = True

        for tgt_ids, log_prob in beams:
            if tgt_ids[-1] == EOS_IDX:
                candidates.append((tgt_ids, log_prob))
                continue

            all_completed = False
            tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(DEVICE)
            tgt_mask = model.make_tgt_mask(tgt_tensor)

            with torch.no_grad():
                output = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)

            log_probs = F.log_softmax(output[0, -1], dim=-1)
            topk_probs, topk_ids = log_probs.topk(beam_size)

            for k in range(beam_size):
                next_id = topk_ids[k].item()
                next_prob = topk_probs[k].item()
                candidates.append((tgt_ids + [next_id], log_prob + next_prob))

        if all_completed:
            break

        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]

    best_seq, _ = beams[0]
    out_words = [itos_es.get(idx, "<UNK>") for idx in best_seq[1:] if idx != EOS_IDX]
    return " ".join(out_words)

# Wrapper estándar para inferencia
def translate_sentence(sentence):
    return translate_beam_search(sentence, beam_size=3, max_len=30)

# Pruebas de verificación rápida
print("=== PRUEBAS CON BEAM SEARCH (K=3) ===")
for text in ["I love learning machine learning.", "What time is it?", "The weather is very nice today."]:
    print(f"EN: {text}")
    print(f"ES: {translate_sentence(text)}\n")

=== PRUEBAS CON BEAM SEARCH (K=3) ===
EN: I love learning machine learning.
ES: Me encanta estudiar .

EN: What time is it?
ES: ¿ A qué hora es ?

EN: The weather is very nice today.
ES: Hoy hace mucho tiempo .



# Evaluación Cuantitativa con Métrica BLEU en el Test Set

In [3]:
# ============================================
# 3. EVALUACIÓN DE BLEU EN EL CONJUNTO DE TEST
# ============================================

def ngram_counts(tokens, n):
    return collections.Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def sentence_bleu(reference, hypothesis, max_n=4):
    ref_tokens = tokenize(reference)
    hyp_tokens = tokenize(hypothesis)

    if len(hyp_tokens) == 0:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = ngram_counts(ref_tokens, n)
        hyp_ngrams = ngram_counts(hyp_tokens, n)

        overlap = sum(min(c, ref_ngrams.get(g, 0)) for g, c in hyp_ngrams.items())
        total = max(sum(hyp_ngrams.values()), 1)
        precisions.append(overlap / total if total > 0 else 1e-9)

    if min(precisions) <= 0:
        geo_mean = 0.0
    else:
        geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)

    bp = 1.0 if len(hyp_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))
    return geo_mean * bp

# Cargar dataset de prueba
test_path = os.path.join(SPLIT_DIR, "test.tsv")
test_pairs = []

with open(test_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 2:
            test_pairs.append((parts[0], parts[1]))

print(f"Evaluando {len(test_pairs):,} ejemplos de test...")

results = []
for src, tgt in test_pairs:
    pred = translate_sentence(src)
    score = sentence_bleu(tgt, pred)
    results.append({
        "english": src,
        "spanish_reference": tgt,
        "spanish_predicted": pred,
        "bleu": score
    })

df_res = pd.DataFrame(results)

print("=" * 60)
print("EVALUACIÓN EN CONJUNTO DE TEST COMPLETADA")
print("=" * 60)
print(f"Total de ejemplos evaluados : {len(df_res):,}")
print(f"BLEU Promedio Global       : {df_res['bleu'].mean():.4f}")

# Guardar informe
REPORTE_DIR = os.path.join(PROJ, "reporte")
os.makedirs(REPORTE_DIR, exist_ok=True)
df_res.to_csv(os.path.join(REPORTE_DIR, "resultados_test.csv"), index=False, encoding="utf-8")
print(f"Resultados guardados en: {os.path.join(REPORTE_DIR, 'resultados_test.csv')}")

Evaluando 7,206 ejemplos de test...
EVALUACIÓN EN CONJUNTO DE TEST COMPLETADA
Total de ejemplos evaluados : 7,206
BLEU Promedio Global       : 0.2474
Resultados guardados en: /content/drive/MyDrive/Proyecto_traductor/reporte/resultados_test.csv


# Análisis Cualitativo de Errores y Clasificación de Ejemplos

In [4]:
# ============================================
# 4. ANÁLISIS CUALITATIVO (MUESTRAS)
# ============================================

print("=== TOP 10 TRADUCCIONES DE BLEU ALTO ===")
display(df_res.sort_values("bleu", ascending=False).head(10)[["english", "spanish_reference", "spanish_predicted", "bleu"]])

print("\n=== TOP 10 TRADUCCIONES DE BLEU BAJO O CON DISCREPANCIAS ===")
display(df_res.sort_values("bleu", ascending=True).head(10)[["english", "spanish_reference", "spanish_predicted", "bleu"]])

print("\n=== MUESTRA DE 100 EJEMPLOS DE TEST ===")
display(df_res.head(100)[["english", "spanish_reference", "spanish_predicted", "bleu"]])

=== TOP 10 TRADUCCIONES DE BLEU ALTO ===


,english,spanish_reference,spanish_predicted,bleu
13,Who ran?,¿Quién corrió?,¿ Quién corrió ?,1.0
11,I'm fit.,Estoy en forma.,Estoy en forma .,1.0
37,Who cares?,¿A quién le importa?,¿ A quién le importa ?,1.0
29,Is it red?,¿Es rojo?,¿ Es rojo ?,1.0
27,I'm a man.,Soy un hombre.,Soy un hombre .,1.0
52,I am a man.,Soy un hombre.,Soy un hombre .,1.0
38,Who drove?,¿Quién condujo?,¿ Quién condujo ?,1.0
40,Am I right?,¿Tengo razón?,¿ Tengo razón ?,1.0
69,I'm a hero.,Soy un héroe.,Soy un héroe .,1.0
7069,Tom didn't have the courage to admit that he h...,Tom no tuvo el valor de admitir que había come...,Tom no tuvo el valor de admitir que había come...,1.0



=== TOP 10 TRADUCCIONES DE BLEU BAJO O CON DISCREPANCIAS ===


,english,spanish_reference,spanish_predicted,bleu
5820,"All of a sudden, all the lights went out.","De repente, todas las luces se apagaron.",Todas las luces se fueron .,0.0
5813,Why do you always ask so many questions?,¿Por qué haces siempre tantas preguntas?,¿ Por qué siempre preguntas tanto ?,0.0
5815,Yesterday I spent the whole day working.,Ayer me quedé trabajando todo el día.,Ayer pasé todo el día trabajando .,0.0
5817,You will live longer if you don't smoke.,"Si no fumas, vivirás más.",Ya no vas a fumar .,0.0
5819,"""Whose shoes are these?"" ""They're Tom's.""","""¿A quién pertenecen estos zapatos?"" ""A Tom.""",""" ¿ De quién son estos zapatos de Tom ? """,0.0
3170,Tom is on a winning streak.,Tom disfruta de una racha de suerte.,Tom está ganando en un ganar .,0.0
3169,Tom is on a winning streak.,Tom tiene una racha de suerte.,Tom está ganando en un ganar .,0.0
3168,Tom is on a winning streak.,Tom está en una racha de suerte.,Tom está ganando en un ganar .,0.0
3167,Tom is likely to be lonely.,Es probable que Tom se sienta solo.,Tom es probable que esté solo .,0.0
3166,Tom is angry and disgusted.,Tom está enfadado e indignado.,Tom está enojado y enojado .,0.0



=== MUESTRA DE 100 EJEMPLOS DE TEST ===


,english,spanish_reference,spanish_predicted,bleu
0,Hop in.,Métete adentro.,Entra .,0.0
1,I'm 19.,Tengo diecinueve.,Me muero .,0.0
2,Ask him.,Pregúntale.,Pregúntale .,0.0
3,Be cool.,Estate tranquilo.,Sé fresco .,0.0
4,He came.,Él vino.,Él vino .,0.0
...,...,...,...,...
95,Don't worry.,No te preocupes.,No te preocupes .,1.0
96,Don't worry.,No os preocupéis.,No te preocupes .,0.0
97,Don't worry.,No se preocupen.,No te preocupes .,0.0
98,Help me out.,Ayúdame.,Ayúdame .,0.0


# Interfaz Web Interactiva en Vivo (Gradio)

In [5]:
# ============================================
# 5. DEMO INTERACTIVA (GRADIO)
# ============================================

!pip install gradio -q
import gradio as gr

def interfaz_traductor(texto_ingles):
    if not texto_ingles or not texto_ingles.strip():
        return ""
    return translate_sentence(texto_ingles)

demo = gr.Interface(
    fn=interfaz_traductor,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Escribe una oración en inglés...",
        label="Texto en Inglés"
    ),
    outputs=gr.Textbox(label="Traducción al Español (Transformer + Beam Search)"),
    title="Traductor Neural Inglés ➔ Español",
    description="Modelo Seq2Seq Transformer implementado desde cero en PyTorch.",
    examples=[
        ["I love learning machine learning."],
        ["What time is it?"],
        ["The weather is very nice today."],
        ["She needs her glasses."]
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9ab04c783f2f82f878.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9ab04c783f2f82f878.gradio.live


In [6]:
oracion_ingles = "I love learning machine learning."
referencia_espanol = "Me encanta aprender aprendizaje automático."

# Generar la traducción con el modelo cargado
prediccion_espanol = translate_sentence(oracion_ingles)

# Calcular su BLEU individual
puntaje_bleu = sentence_bleu(referencia_espanol, prediccion_espanol)

print(f"Inglés     : {oracion_ingles}")
print(f"Referencia : {referencia_espanol}")
print(f"Predicción : {prediccion_espanol}")
print(f"BLEU       : {puntaje_bleu:.4f}")

Inglés     : I love learning machine learning.
Referencia : Me encanta aprender aprendizaje automático.
Predicción : Me encanta estudiar .
BLEU       : 0.0000
